In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

In [ ]:
import torch as tr
import torch.nn as nn
import torch.autograd as autograd

import import_ipynb
import bce # type: ignore
import linear # type: ignore
import sigmoid # type: ignore

from common import test_checkgrad_2, test_compare_2 # type: ignore
from approx import approx # type: ignore

In [ ]:
class Per_Lin_Sig_BCE_Autograd(nn.Module):
    """The version relying fully on PyTorch to handle `forward` and `backward` passes."""
        
    def __init__(self, in_features, out_features):
        super().__init__()

        #
        # [Linear  + [BinaryCrossEntropyWithLogits = Sigmoid + BinaryCrossEntropy] 
        # is more numerically stable than [Linear] + [Sigmoid] + [BinaryCrossEntropy]
        #

        self.linear =  nn.Linear(in_features, out_features)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        z = self.model(x)
        return z
    
    def model(self, x):
        z = self.linear(x)
        p = self.sigmoid(z)
        return p

    def loss(self, p, y):
        L = nn.BCELoss(reduction='mean')(p, y)
        return L

    def predict(self, x):
        with tr.no_grad():
            y = (self.model(x) > 0.5).float()
            return y

In [ ]:
class Per_Lin_Sig_BCE_Backward(nn.Module):
    """
    The version where the `forward` and `backward` passes for each operation are written manually, 
    but PyTorch’s autograd still orchestrates the overall gradient flow through the computational graph.
    """

    def __init__(self, in_features: int, out_features: int, W: tr.Tensor = None, b: tr.Tensor = None) -> None:
        super().__init__()

        #
        # [Linear Layer] + [BinaryCrossEntropyWithLogits = Sigmoid + BinaryCrossEntropy] 
        # is more numerically stable than [Linear] + [Sigmoid] + [BinaryCrossEntropy]
        #

        self.linear =  linear.Linear(in_features, out_features, W, b)
        self.sigmoid = sigmoid.Sigmoid()

    def model(self, x):
        z = self.linear(x)
        p = self.sigmoid(z)
        return p
    
    def loss(self, p, y):
        L = bce.BCE(reduction='mean')(p, y)
        return L

    def predict(self, x):
        with tr.no_grad():
            y = (self.model(x) > 0.5).float()
            return y
        
    def forward(self, x):
        z = self.model(x)
        return z

$$ m\text{odel function} $$

$$ m:\mathbb{R} \times \mathbb{R} \to \mathbb{R}, \quad z=m(w, b) = xw + b $$

$$ 
m: \mathbb{R}^m \times \mathbb{R}^n \to \mathbb{R}^n, \quad \mathbf{z}=
m(\mathbf{w}, \mathbf{b}) = X\mathbf{w} + \mathbf{b}
$$

$$ \frac{\partial m}{\partial \mathbf{w}} = X $$

$$ \frac{\partial m}{\partial \mathbf{b}} = \mathbf{I}_{n} $$

$$ 
d\mathbf{z} = 
\frac{\partial m}{\partial \mathbf{w}} d\mathbf{w} + \frac{\partial m}{\partial \mathbf{b}} d\mathbf{b} 
$$

$$ a\text{ctivation function} $$

$$ a:\mathbb{R} \to \mathbb{R}, \quad p = a(z) = \frac{e^z}{1+e^z} $$

$$ 
a:\mathbb{R^n} \to \mathbb{R^n}, \quad \mathbf{p} = 
a(\mathbf{z}) = 
(a(z_1), a(z_2), \dots, a(z_n)) 
$$

$$
\frac{d a}{dz} = p(1 - p_i)
$$


$$ \mathbb{R} \to \mathbb{R}, \quad p(z) = \frac{e^z}{1+e^z} $$

$$ 
\mathbb{R^n} \to \mathbb{R^n}, \quad p(\mathbf{z}) =
(p(z_1), p(z_2), \dots, p(z_n)) 
$$

$$
\frac{dp}{dz} = p(1 - p)
$$

$$ L\text{oss function} $$

$$ 
L:(0, 1)^n \times \{0, 1\}^n \to \mathbb{R}, \quad \mathbf{y} = 
L(\mathbf{p}, \mathbf{t}) = 
-\frac{1}{N} \sum \Big(t _i \ln(p_i) + (1 - t_i) \ln(1 - p_i) \Big) 
$$

$$
\frac{\partial L}{\partial p} = 
\frac{1}{N} \frac{p - t}{p(1 - p)} 
$$

$$ 
\frac{\partial L}{\partial z} = 
\frac{\partial L}{\partial p} \frac{\partial p}{\partial z} = p-t $$



$$ \frac{\partial L}{\partial z} = \frac{\partial L}{\partial p} \frac{\partial p}{\partial z} = p-t $$

$$ \frac{\partial L}{\partial W} = \frac{\partial L}{\partial z} \frac{\partial z}{\partial W} = x^T \cdot (p-t) $$

$$ \frac{\partial L}{\partial b} = \frac{\partial L}{\partial z} \frac{\partial z}{\partial b} = (p-t) \cdot 1 $$


In [ ]:
class Per_Lin_Sig_BCE_Gradient_Function(autograd.Function):
    @staticmethod
    def __model(x, W, b):
        z = linear.linear(x, W, b)
        p = sigmoid.sigmoid(z)
        return p
    
    @staticmethod
    def __loss(p, y):
        L = bce.bce(p, y)
        return L

    @staticmethod
    def predict(x, W, b):
        with tr.no_grad():
            y = (Per_Lin_Sig_BCE_Gradient_Function.__model(x, W, b) >= 0.5).float()
            return y

    @staticmethod
    def forward(ctx, x, W, b, t):
        p = Per_Lin_Sig_BCE_Gradient_Function.__model(x, W, b)
        L = Per_Lin_Sig_BCE_Gradient_Function.__loss(p, t)

        ctx.save_for_backward(x, W, t, p)

        return L
    
    @staticmethod
    def backward(ctx, dF_df):
        x = ctx.saved_tensors[0]
        W = ctx.saved_tensors[1]
        t = ctx.saved_tensors[2]
        p = ctx.saved_tensors[3]
        
        S = x.shape[0]  # Samples
        FI = x.shape[1] # Features In
        FO = W.shape[1] # Features Out
        N = S * FO

        dz_dw = x
        dz_db = tr.ones_like(p, dtype=tr.float64)
        
        dL_dz = 1/N * (p - t)
        dF_dW = dF_df * dz_dw.t() @ dL_dz
        dF_db = dF_df * dz_db.t() @ dL_dz

        return (None, dF_dW, dF_db, None)
    

class Per_Lin_Sig_BCE_Gradient(nn.Module):
    """
    The version exposing the exact mechanics of gradient computation and giving 
    full control over how the model participates in PyTorch's autograd system.
    """

    class _Linear:
        """Helper class to keep the model internal layout consistent across all variants."""

        def __init__(self, w, b):
            self.weight = w
            self.bias = b

    # This is helper for testing to indicate that the `forward` method expects both, `x` and `t`.
    CUSTOM_GRAD=True

    def __init__(self, in_features: int, out_features: int, W: tr.Tensor = None, b: tr.Tensor = None) -> None:
        super().__init__()

        # `W` has shape (out_features, in_features) to be consistent with `nn.Linear`
        if W is None:
            self.weight = nn.Parameter(tr.randn(out_features, in_features, dtype=tr.float32))
        else:
            self.weight = nn.Parameter(W.clone().detach().requires_grad_(True))

        if b is None:
            self.bias = nn.Parameter(tr.randn(out_features, dtype=tr.float32))
        else:
            self.bias = nn.Parameter(b.clone().detach().requires_grad_(True))

        self.linear = Per_Lin_Sig_BCE_Gradient._Linear(self.weight, self.bias)

    def model(self, x):
        with tr.no_grad():
            z = Per_Lin_Sig_BCE_Gradient_Function.__model(x, self.weight.t(), self.bias)
            return z
    
    def loss(self, p, t):
        with tr.no_grad():
            L = Per_Lin_Sig_BCE_Gradient_Function.__loss(p, t)
            return L

    def predict(self, x):
        with tr.no_grad():
            y = Per_Lin_Sig_BCE_Gradient_Function.predict(x, self.weight.t(), self.bias)
            return y
        
    def forward(self, x, t):
        y = Per_Lin_Sig_BCE_Gradient_Function.apply(x, self.weight.t(), self.bias, t)
        return y

In [ ]:
import import_ipynb
from common import test_model_sgd_2, repeat # type: ignore

def test_logical_fn(model, bool_fn, samples=10, epochs=500, lr=0.1):
    x = (tr.randn(samples, 2, dtype=tr.float32) > 0).float()
    t = bool_fn(x[:, 0], x[:, 1]).float().unsqueeze(1)
    return test_model_sgd_2(model, x, t, epochs, lr=lr)

if __name__ == "__main__":
    test_checkgrad_2(Per_Lin_Sig_BCE_Gradient_Function, 1, 1, 1)
    test_checkgrad_2(Per_Lin_Sig_BCE_Gradient_Function, 2, 2, 2)
    test_checkgrad_2(Per_Lin_Sig_BCE_Gradient_Function, 3, 3, 3)

    test_compare_2(Per_Lin_Sig_BCE_Autograd, Per_Lin_Sig_BCE_Backward, 1, 1, 1)
    test_compare_2(Per_Lin_Sig_BCE_Autograd, Per_Lin_Sig_BCE_Backward, 10, 1, 1)
    test_compare_2(Per_Lin_Sig_BCE_Autograd, Per_Lin_Sig_BCE_Backward, 10, 20, 1)
    test_compare_2(Per_Lin_Sig_BCE_Autograd, Per_Lin_Sig_BCE_Backward, 10, 20, 30)

    test_compare_2(Per_Lin_Sig_BCE_Autograd, Per_Lin_Sig_BCE_Gradient, 1, 1, 1)
    test_compare_2(Per_Lin_Sig_BCE_Autograd, Per_Lin_Sig_BCE_Gradient, 10, 1, 1)
    test_compare_2(Per_Lin_Sig_BCE_Autograd, Per_Lin_Sig_BCE_Gradient, 10, 20, 1)
    test_compare_2(Per_Lin_Sig_BCE_Autograd, Per_Lin_Sig_BCE_Gradient, 10, 20, 30)